# Predictive Maintenance: Machine Failure Prediction using Naïve Bayes

**Course project: Machine Learning Pipeline (Supervised Learning, Classification)**

## Problem Statement
In industry, an unexpected machine breakdown causes costly downtime, while unnecessary maintenance wastes money and labour.
Using four sensor readings from a milling machine, **air/process temperature, rotational speed, torque and tool wear**, build a
**Naïve Bayes classifier** that predicts whether the machine will **fail (1)** or **keep running normally (0)**.

## Dataset
**AI4I 2020 Predictive Maintenance Dataset** (UCI Machine Learning Repository), 10,000 rows, original sensor features and a binary label `Machine failure`.
- Source: https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset
- Same CSV on Kaggle: search *"AI4I 2020 Predictive Maintenance Dataset"* → file `ai4i2020.csv`

## Pipeline
1. Load CSV → 2. Explore data → 3. Select sensor features → 4. **80-20 stratified train/test split** → 5. Scale →
6. Train Gaussian Naïve Bayes → 7. Train **Kernel (KDE) Naïve Bayes** (kernel-trick version) → 8. Evaluate (Accuracy, Confusion Matrix, Type I/II error, Precision, Recall, F1, Heat map) → 9. Threshold tuning → 10. Discussion

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KernelDensity
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, classification_report)
from scipy.special import logsumexp

sns.set_style("whitegrid")

## 2. Load the dataset (.csv)
Download `ai4i2020.csv` from the UCI/Kaggle link above, then run the cell below and select the file when asked.

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose ai4i2020.csv
df = pd.read_csv(list(uploaded.keys())[0])
df.head()

## 3. Explore the data

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing values:\n", df.isnull().sum().sum())
df.describe().T

In [ ]:
# Column names differ slightly between the Kaggle and UCI versions (e.g. 'Air temperature [K]' vs 'Air temperature'),
# so we locate the columns by keyword.
def find_col(keyword):
    return [c for c in df.columns if keyword.lower() in c.lower()][0]

feature_cols = [find_col("Air temperature"), find_col("Process temperature"),
                find_col("Rotational speed"), find_col("Torque"), find_col("Tool wear")]
target_col = find_col("Machine failure")

print("Features:", feature_cols)
print("Target  :", target_col)

counts = df[target_col].value_counts()
print("\nClass balance:\n", counts)
print(f"Failure rate = {counts[1] / len(df) * 100:.2f}%  (highly imbalanced)")

plt.figure(figsize=(4, 3))
sns.countplot(x=target_col, data=df)
plt.title("Class distribution (0 = Normal, 1 = Failure)")
plt.show()

In [ ]:
# Sensor distributions for normal vs failed machines
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, col in zip(axes, feature_cols):
    sns.kdeplot(data=df, x=col, hue=target_col, common_norm=False, fill=True, ax=ax)
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.heatmap(df[feature_cols + [target_col]].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heat map (sensors vs failure)")
plt.show()

## 4. Feature selection and 80-20 train/test split
- We use **only the 5 sensor features** (temperature, speed, torque, tool wear).
- `UDI`, `Product ID` are identifiers. `Type` is categorical. **`TWF, HDF, PWF, OSF, RNF` are the failure-mode columns, which directly encode the label (data leakage), so they are deliberately excluded.**
- `stratify=y` keeps the same failure ratio (~3.4%) in both train and test sets.

In [ ]:
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Failures in train:", y_train.sum(), "| Failures in test:", y_test.sum())

# Scale (fit on TRAIN only to avoid leakage)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

## 5. Evaluation helper
Convention used (failure = positive class):

| | Predicted Normal (0) | Predicted Failure (1) |
|---|---|---|
| **Actual Normal (0)** | TN | **FP → Type I error** (false alarm) |
| **Actual Failure (1)** | **FN → Type II error** (missed failure) | TP |

- **Type I error rate (α)** = FP / (FP + TN)
- **Type II error rate (β)** = FN / (FN + TP) = 1 − Recall

In [ ]:
def evaluate(name, y_true, y_pred, ax=None, show_report=True):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    res = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
        "Type I error (FPR)": fp / (fp + tn),
        "Type II error (FNR)": fn / (fn + tp),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }
    print("=" * 55, f"\n{name}\n" + "=" * 55)
    print(f"Accuracy            : {res['Accuracy']:.4f}")
    print(f"Precision           : {res['Precision']:.4f}")
    print(f"Recall (Sensitivity): {res['Recall']:.4f}")
    print(f"F1-score            : {res['F1-score']:.4f}")
    print(f"Type I error  (FP)  : {fp} cases -> rate = {res['Type I error (FPR)']:.4f}  (false alarms)")
    print(f"Type II error (FN)  : {fn} cases -> rate = {res['Type II error (FNR)']:.4f}  (missed failures)")
    if show_report:
        print("\n", classification_report(y_true, y_pred, target_names=["Normal", "Failure"], zero_division=0))

    # Heat map of confusion matrix
    cm = np.array([[tn, fp], [fn, tp]])
    labels = np.array([[f"TN\n{tn}", f"FP (Type I)\n{fp}"],
                       [f"FN (Type II)\n{fn}", f"TP\n{tp}"]])
    if ax is None:
        plt.figure(figsize=(5, 4)); ax = plt.gca(); standalone = True
    else:
        standalone = False
    sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Pred Normal", "Pred Failure"],
                yticklabels=["Actual Normal", "Actual Failure"])
    ax.set_title(f"Confusion Matrix Heat Map\n{name}")
    if standalone:
        plt.tight_layout(); plt.show()
    return res

## 6. Model 1: Gaussian Naïve Bayes
Assumes each sensor reading, *given the class*, follows a Gaussian (normal) distribution and that sensors are conditionally independent:

$$P(\text{Failure}\mid x_1..x_5) \propto P(\text{Failure})\prod_{i=1}^{5} P(x_i \mid \text{Failure})$$

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train_s, y_train)
y_pred_gnb = gnb.predict(X_test_s)

res_gnb = evaluate("Gaussian Naive Bayes (threshold 0.5)", y_test, y_pred_gnb)

## 7. Model 2: Kernel Naïve Bayes (kernel trick)
Gaussian NB forces a bell-curve on every feature. Here the **kernel trick idea is applied to Naïve Bayes via Kernel Density Estimation (KDE)**:
each class-conditional density is estimated as a sum of kernel functions placed on every training point,

$$\hat p(x_i\mid c)=\frac{1}{n_c h}\sum_{j=1}^{n_c}K\!\left(\frac{x_i-x_{ij}}{h}\right)$$

so the model can learn **non-Gaussian, skewed or multi-modal** sensor distributions (e.g. torque of failing machines).

In [ ]:
class KernelNaiveBayes(BaseEstimator, ClassifierMixin):
    """Naive Bayes whose per-feature class-conditional densities are estimated with KDE (kernel trick)."""
    def __init__(self, bandwidth=0.3, kernel="gaussian"):
        self.bandwidth = bandwidth
        self.kernel = kernel

    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        self.classes_ = np.unique(y)
        self.log_priors_ = np.log(np.array([(y == c).mean() for c in self.classes_]))
        self.kdes_ = []
        for c in self.classes_:
            Xc = X[y == c]
            self.kdes_.append([KernelDensity(bandwidth=self.bandwidth, kernel=self.kernel).fit(Xc[:, [j]])
                               for j in range(X.shape[1])])
        return self

    def _joint_log_likelihood(self, X):
        X = np.asarray(X)
        jll = np.zeros((X.shape[0], len(self.classes_)))
        for k in range(len(self.classes_)):
            jll[:, k] = self.log_priors_[k] + sum(
                self.kdes_[k][j].score_samples(X[:, [j]]) for j in range(X.shape[1]))
        return jll

    def predict_proba(self, X):
        jll = self._joint_log_likelihood(X)
        return np.exp(jll - logsumexp(jll, axis=1, keepdims=True))

    def predict(self, X):
        return self.classes_[np.argmax(self._joint_log_likelihood(X), axis=1)]


knb = KernelNaiveBayes(bandwidth=0.3, kernel="gaussian")
knb.fit(X_train_s, y_train)
y_pred_knb = knb.predict(X_test_s)

res_knb = evaluate("Kernel (KDE) Naive Bayes (threshold 0.5)", y_test, y_pred_knb)

## 7A. Model 3 (added): AdaBoost Classifier
**AdaBoost (Adaptive Boosting)** is an *ensemble* method. It trains many very small decision trees (weak learners, "stumps") one after another.
After each round, the training samples that were **misclassified get a higher weight**, so the next weak learner concentrates on the hard cases (here, the rare failures).
The final prediction is a weighted vote of all weak learners:

$$F(x)=\text{sign}\Big(\sum_{t=1}^{T}\alpha_t\,h_t(x)\Big)$$

where $h_t$ is the t-th weak learner and $\alpha_t$ is its weight (larger when its weighted error is smaller).
It is added here **for comparison** with Gaussian NB and Kernel NB. Unlike Naïve Bayes it makes **no independence assumption** between sensors.
(AdaBoost is not a kernel method, so no kernel is used here.)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42)
ada.fit(X_train_s, y_train)
y_pred_ada = ada.predict(X_test_s)

res_ada = evaluate("AdaBoost Classifier (threshold 0.5)", y_test, y_pred_ada)

In [ ]:
# Which sensors does AdaBoost rely on most?
imp = pd.Series(ada.feature_importances_, index=feature_cols).sort_values()
imp.plot(kind="barh", figsize=(7, 3), color="purple")
plt.title("AdaBoost feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 8. Reducing Type II error: decision-threshold tuning
Default NB predicts "Failure" only if P(Failure) > 0.5. In maintenance, **missing a failure (Type II) is far costlier than a false alarm (Type I)**,
so we lower the threshold. To avoid cheating, the threshold is **chosen using cross-validation on the TRAINING set only**
(maximising F2-score, which weights recall higher than precision) and then applied once to the test set.

In [ ]:
from sklearn.metrics import fbeta_score

def tune_threshold(model, Xtr, ytr):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    proba = cross_val_predict(model, Xtr, ytr, cv=cv, method="predict_proba")[:, 1]
    thresholds = np.linspace(0.01, 0.99, 99)
    scores = [fbeta_score(ytr, (proba >= t).astype(int), beta=2, zero_division=0) for t in thresholds]
    return thresholds[int(np.argmax(scores))]

thr_gnb = tune_threshold(GaussianNB(), X_train_s, y_train)
thr_knb = tune_threshold(KernelNaiveBayes(bandwidth=0.3), X_train_s, y_train)
print(f"Chosen threshold  Gaussian NB: {thr_gnb:.2f} | Kernel NB: {thr_knb:.2f}")

y_pred_gnb_t = (gnb.predict_proba(X_test_s)[:, 1] >= thr_gnb).astype(int)
y_pred_knb_t = (knb.predict_proba(X_test_s)[:, 1] >= thr_knb).astype(int)

res_gnb_t = evaluate(f"Gaussian NB (tuned threshold {thr_gnb:.2f})", y_test, y_pred_gnb_t)
res_knb_t = evaluate(f"Kernel NB (tuned threshold {thr_knb:.2f})", y_test, y_pred_knb_t)

## 8A. AdaBoost: decision-threshold tuning
Same procedure as Section 8: the threshold is chosen by cross-validation on the **training set only** (maximising F2-score) and then applied once to the test set.

In [ ]:
thr_ada = tune_threshold(AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42), X_train_s, y_train)
print(f"Chosen threshold AdaBoost: {thr_ada:.2f}")

y_pred_ada_t = (ada.predict_proba(X_test_s)[:, 1] >= thr_ada).astype(int)
res_ada_t = evaluate(f"AdaBoost (tuned threshold {thr_ada:.2f})", y_test, y_pred_ada_t)

## 9. Side-by-side comparison (all four heat maps + metrics table)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
results = []
for ax, (nm, yp) in zip(axes.ravel(), [
        ("Gaussian NB (0.5)", y_pred_gnb),
        (f"Gaussian NB (thr {thr_gnb:.2f})", y_pred_gnb_t),
        ("Kernel NB (0.5)", y_pred_knb),
        (f"Kernel NB (thr {thr_knb:.2f})", y_pred_knb_t)]):
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):       # silence repeated printing
        results.append(evaluate(nm, y_test, yp, ax=ax, show_report=False))
plt.tight_layout()
plt.show()

summary = pd.DataFrame(results).set_index("Model")
summary.round(4)

In [ ]:
# Bar chart: Type I vs Type II error for each model
summary[["Type I error (FPR)", "Type II error (FNR)"]].plot(kind="bar", figsize=(9, 4))
plt.title("Type I vs Type II error rate")
plt.ylabel("Error rate"); plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.show()

## 9A. Final comparison of all three classifiers (Gaussian NB, Kernel NB, AdaBoost)

In [ ]:
import io, contextlib

all_preds = [
    ("Gaussian NB (0.5)",               y_pred_gnb),
    (f"Gaussian NB (thr {thr_gnb:.2f})", y_pred_gnb_t),
    ("Kernel NB (0.5)",                 y_pred_knb),
    (f"Kernel NB (thr {thr_knb:.2f})",   y_pred_knb_t),
    ("AdaBoost (0.5)",                  y_pred_ada),
    (f"AdaBoost (thr {thr_ada:.2f})",    y_pred_ada_t),
]

fig, axes = plt.subplots(3, 2, figsize=(11, 13))
results_all = []
for ax, (nm, yp) in zip(axes.ravel(), all_preds):
    with contextlib.redirect_stdout(io.StringIO()):
        results_all.append(evaluate(nm, y_test, yp, ax=ax, show_report=False))
plt.tight_layout()
plt.show()

summary_all = pd.DataFrame(results_all).set_index("Model")
summary_all.round(4)

In [ ]:
# Type I vs Type II error for all three classifiers
summary_all[["Type I error (FPR)", "Type II error (FNR)"]].plot(kind="bar", figsize=(10, 4), color=["gold", "red"])
plt.title("Type I vs Type II error rate (all classifiers)")
plt.ylabel("Error rate"); plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.show()

# Accuracy / Precision / Recall / F1
summary_all[["Accuracy", "Precision", "Recall", "F1-score"]].plot(kind="bar", figsize=(10, 4),
                                                                color=["purple", "green", "gold", "red"])
plt.title("Accuracy, Precision, Recall, F1 (all classifiers)")
plt.xticks(rotation=20, ha="right"); plt.ylim(0, 1.05)
plt.tight_layout(); plt.show()

## 9B. Discussion: AdaBoost vs Naïve Bayes (check against your own numbers)
1. **Independence assumption:** Naïve Bayes assumes sensors are independent given the class; AdaBoost does not, so it can use interactions such as the torque-speed relationship through its trees. This is the main reason it is worth comparing.
2. **Focus on hard cases:** boosting re-weights misclassified samples, so the rare failure class receives more attention over the training rounds.
3. **Trade-offs:** AdaBoost is slower to train and less interpretable than Naïve Bayes, and its probability scores are less well calibrated, so the decision threshold should be tuned (Section 8A).
4. **How to choose:** for predictive maintenance, prefer the model and threshold with the lowest **Type II error** that still keeps **Type I error** acceptable, and support the choice with F1-score.

## 10. Discussion: Type I vs Type II error
| Error | Meaning here | Real-world consequence |
|---|---|---|
| **Type I (False Positive)** | Model predicts *failure* but the machine is healthy | Unnecessary inspection / downtime, small cost |
| **Type II (False Negative)** | Model predicts *normal* but the machine fails | Unplanned breakdown, production loss, safety risk, **high cost** |

**Key observations (check them against your own numbers):**
1. **Accuracy is misleading.** Only ~3.4% of rows are failures, so a model that always says "Normal" gets ~96.6% accuracy yet has a 100% Type II error. This is why Precision, Recall, F1 and the confusion matrix are required.
2. **Trade-off:** lowering the decision threshold reduces Type II error (higher recall) but increases Type I error (lower precision). Both cannot be minimised together.
3. **Why Naïve Bayes struggles a little:** it assumes the sensors are independent given the class, but in reality *torque and rotational speed are strongly (negatively) correlated* and temperatures are correlated too. This assumption violation limits precision.
4. **Kernel NB vs Gaussian NB:** the KDE version does not force a normal shape, so it can capture skewed sensor distributions of failing machines, which often improves recall/F1 at the cost of more computation.
5. **For predictive maintenance, prefer the model/threshold with the lowest Type II error that still keeps Type I error acceptable.**

## 11. Conclusion and future work
- Naïve Bayes gives a fast, simple and interpretable baseline for sensor-based failure prediction.
- Threshold tuning shifted the model toward catching more failures, which matches the business cost of Type II errors.
- **Future work:** handle imbalance using SMOTE / class weights, engineer features (e.g. power = torque × rpm, temperature difference = process − air), compare with SVM (RBF kernel), Random Forest, and use cross-validation.